# Constant-frequency DummyRegressor baseline (Poisson example): scikit-learn vs Moxie

**Reference example:** `../scikit-learn/examples/linear_model/plot_poisson_regression_non_normal_loss.py` at
scikit-learn `4358542a1bd39af2e7e1cbace8b9fed067473c5b`
([rendered gallery page](https://scikit-learn.org/dev/auto_examples/linear_model/plot_poisson_regression_non_normal_loss.html)).

**Scenario:** the example's first model is a constant baseline,
`Pipeline([("preprocessor", ColumnTransformer(...)), ("regressor", DummyRegressor(strategy="mean"))])`,
fitted with `regressor__sample_weight=Exposure` and scored with exposure-weighted MSE, MAE and mean Poisson deviance.

**Adaptations:**
- The example downloads freMTPL2freq from OpenML. This notebook does not download: it simulates a dataset with the
  same column names (`VehBrand`, `VehPower`, `VehGas`, `Region`, `Area`, `VehAge`, `DrivAge`, `BonusMalus`, `Density`,
  `Exposure`, `ClaimNb`) and Poisson claim counts, 20,000 rows, seed 0. The example's preprocessor is used unchanged.
- Moxie's `Pipeline.fit` has no sample-weight routing (documented as deferred in `docs/pipeline/sequential.md`),
  so `regressor__sample_weight` cannot be expressed. The weighted baseline is fitted on `DummyRegressor` directly,
  which is equivalent because the estimator ignores X. The pipeline itself is exercised unweighted (Moxie's
  `ColumnTransformer` passes the numeric columns through; the dummy output does not depend on features) and through
  `Pipeline.score(..., sample_weight=...)`.
- Moxie has no `moxie.metrics`; the three metrics are computed in Python from each side's predictions.
- Extra checks beyond the example: exposure-weighted `median`/`quantile(0.9)` strategies, `constant`, and
  `predict(return_std=True)`.

**Determinism:** deterministic (fixed data seed and split seed, no randomness in DummyRegressor). Moxie receives the
sklearn train/test rows, so predictions and learned constants are compared numerically.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("dummy/plot_poisson_regression_non_normal_loss")

## 1. scikit-learn (Python reference)

The example's preprocessing and DummyRegressor baseline on simulated data.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(0)
n = 20_000
df = pd.DataFrame({
    "VehBrand": rng.choice([f"B{i}" for i in range(1, 7)], n),
    "VehPower": rng.randint(4, 15, n),
    "VehGas": rng.choice(["Diesel", "Regular"], n),
    "Region": rng.choice([f"R{i}" for i in range(11, 25)], n),
    "Area": rng.choice(list("ABCDEF"), n),
    "VehAge": rng.randint(0, 20, n).astype(float),
    "DrivAge": rng.randint(18, 90, n).astype(float),
    "BonusMalus": rng.randint(50, 150, n).astype(float),
    "Density": np.exp(rng.uniform(1, 10, n)),
    "Exposure": rng.uniform(0.05, 1.0, n),
})
rate = 0.05 * np.exp(0.01 * (df["BonusMalus"] - 50) - 0.01 * df["VehAge"])
df["ClaimNb"] = rng.poisson(rate * df["Exposure"])
df["Frequency"] = df["ClaimNb"] / df["Exposure"]
print("Average Frequency = {}".format(np.average(df["Frequency"], weights=df["Exposure"])))

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, KBinsDiscretizer, OneHotEncoder, StandardScaler

log_scale_transformer = make_pipeline(
    FunctionTransformer(np.log, validate=False), StandardScaler()
)

linear_model_preprocessor = ColumnTransformer(
    [
        ("passthrough_numeric", "passthrough", ["BonusMalus"]),
        (
            "binned_numeric",
            KBinsDiscretizer(n_bins=10, quantile_method="averaged_inverted_cdf"),
            ["VehAge", "DrivAge"],
        ),
        ("log_scaled_numeric", log_scale_transformer, ["Density"]),
        (
            "onehot_categorical",
            OneHotEncoder(),
            ["VehBrand", "VehPower", "VehGas", "Region", "Area"],
        ),
    ],
    remainder="drop",
)

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

df_train, df_test = train_test_split(df, test_size=0.33, random_state=0)

dummy = Pipeline(
    [
        ("preprocessor", linear_model_preprocessor),
        ("regressor", DummyRegressor(strategy="mean")),
    ]
).fit(df_train, df_train["Frequency"], regressor__sample_weight=df_train["Exposure"])

from sklearn.metrics import mean_absolute_error, mean_poisson_deviance, mean_squared_error


def metrics(y_true, y_pred, exposure):
    mask = y_pred > 0
    return {
        "MSE": mean_squared_error(y_true, y_pred, sample_weight=exposure),
        "MAE": mean_absolute_error(y_true, y_pred, sample_weight=exposure),
        # A zero constant (e.g. the weighted median of mostly-zero frequencies) leaves nothing to score.
        "mean Poisson deviance": (mean_poisson_deviance(y_true[mask], y_pred[mask], sample_weight=exposure[mask])
                                  if mask.any() else np.nan),
    }


def score_estimator(estimator, df_test):
    # The example prints these three exposure-weighted metrics.
    y_pred = estimator.predict(df_test)
    for name, value in metrics(df_test["Frequency"].to_numpy(), y_pred, df_test["Exposure"].to_numpy()).items():
        print(f"{name}: {value:.3f}")


print("Constant mean frequency evaluation:")
score_estimator(dummy, df_test)

Average Frequency = 0.0788298813326724


Constant mean frequency evaluation:
MSE: 0.141
MAE: 0.148
mean Poisson deviance: 0.471


In [3]:
numeric = ["VehAge", "DrivAge", "BonusMalus", "Density"]
X_train, X_test = df_train[numeric].to_numpy(), df_test[numeric].to_numpy()
y_train, y_test = df_train["Frequency"].to_numpy(), df_test["Frequency"].to_numpy()
w_train, w_test = df_train["Exposure"].to_numpy(), df_test["Exposure"].to_numpy()

sk = {"mean_weighted": dummy[-1]}
sk["median_weighted"] = DummyRegressor(strategy="median").fit(X_train, y_train, sample_weight=w_train)
sk["quantile90_weighted"] = DummyRegressor(strategy="quantile", quantile=0.9).fit(X_train, y_train, sample_weight=w_train)
sk["median"] = DummyRegressor(strategy="median").fit(X_train, y_train)
sk["constant"] = DummyRegressor(strategy="constant", constant=0.1).fit(X_train, y_train)
unweighted_pipe = Pipeline([("preprocessor", linear_model_preprocessor),
                            ("regressor", DummyRegressor(strategy="mean"))]).fit(df_train, y_train)
sk_pred = {name: model.predict(X_test) for name, model in sk.items()}
sk_pipe_pred = unweighted_pipe.predict(df_test)
sk_pipe_score = unweighted_pipe.score(df_test, y_test, sample_weight=w_test)
sk_std = sk["mean_weighted"].predict(X_test, return_std=True)[1]
nb.share(X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, w_train=w_train, w_test=w_test)

## 2. Moxie (native Mojo)

In [4]:
%%mojo dummy_regressor
from moxie.compose import ColumnTransformer, ColumnEntry, NoTransformer
from moxie.dummy import DummyRegressor
from moxie.pipeline import Pipeline
from moxie_nb_io import load, save, save_floats
from numojo import NDArray

comptime Dense = NDArray[DType.float64]


def main() raises:
    var X_train = load("X_train")
    var X_test = load("X_test")
    var y_train = load("y_train")
    var y_test = load("y_test")
    var w_train = load("w_train")
    var w_test = load("w_test")

    var mean = DummyRegressor(strategy="mean")
    _ = mean.fit(X_train, y_train, sample_weight=w_train)
    save("mean_weighted", mean.predict(X_test))
    save("mean_weighted_constant", mean.constant_.value())
    var with_std = mean.predict(X_test, return_std=True)
    save("mean_weighted_std", with_std.std.value())

    var median_w = DummyRegressor(strategy="median")
    _ = median_w.fit(X_train, y_train, sample_weight=w_train)
    save("median_weighted", median_w.predict(X_test))
    var quantile_w = DummyRegressor(strategy="quantile", quantile=0.9)
    _ = quantile_w.fit(X_train, y_train, sample_weight=w_train)
    save("quantile90_weighted", quantile_w.predict(X_test))
    var median = DummyRegressor(strategy="median")
    _ = median.fit(X_train, y_train)
    save("median", median.predict(X_test))
    var constant = DummyRegressor(constant=Float64(0.1), strategy="constant")
    _ = constant.fit(X_train, y_train)
    save("constant", constant.predict(X_test))

    # Unweighted pipeline: ColumnTransformer passthrough + DummyRegressor(mean).
    var columns = ColumnTransformer(
        ColumnEntry[NoTransformer[DType.float64], DType.float64]("numeric", "passthrough", [0, 1, 2, 3]),
        remainder="drop")
    var pipe = Pipeline(columns^, DummyRegressor(strategy="mean"),
                        transformer_name="preprocessor", estimator_name="regressor")
    _ = pipe.fit(X_train, y_train)
    save("pipe_unweighted", pipe.predict(X_test))
    save_floats("pipe_score_weighted", [pipe.score(X_test, y_test, w_test.copy())])
    print("moxie weighted mean frequency:", mean.constant_.value().item(0, 0))

moxie weighted mean frequency: 0.08019689533713441


## 3. Side by side and parity

In [5]:
mx = nb.results["dummy_regressor"]
rows = []
for name in ["mean_weighted", "median_weighted", "quantile90_weighted"]:
    for side, pred in [("sklearn", sk_pred[name]), ("moxie", mx[name])]:
        rows.append({"model": name, "side": side, **metrics(y_test, pred, w_test)})
moxie_nb.side_by_side(metrics=pd.DataFrame(rows).set_index(["model", "side"]).round(6))

nb.compare("weighted mean constant_", sk["mean_weighted"].constant_, mx["mean_weighted_constant"],
           rtol=1e-12, atol=0,
           reason="np.average uses pairwise summation, Moxie a sequential weighted sum (20k terms)")
for name in sk_pred:
    exact = name not in ("mean_weighted",)
    nb.compare(f"predict ({name})", sk_pred[name], mx[name], exact=exact, rtol=1e-12,
               reason="" if exact else "summation order of the weighted mean, as above")
nb.compare("predict(return_std=True) std", sk_std, mx["mean_weighted_std"], exact=True)
nb.compare("unweighted Pipeline(ColumnTransformer, DummyRegressor) predict", sk_pipe_pred, mx["pipe_unweighted"],
           rtol=1e-12, reason="summation order of the unweighted mean")
nb.compare("Pipeline.score with sample_weight (R2)", sk_pipe_score, mx["pipe_score_weighted"][0], rtol=1e-9,
           reason="R2 ratio of weighted sums; summation order differs")
for metric in ["MSE", "MAE", "mean Poisson deviance"]:
    nb.compare(f"{metric} (weighted mean baseline)", metrics(y_test, sk_pred["mean_weighted"], w_test)[metric],
               metrics(y_test, mx["mean_weighted"], w_test)[metric], rtol=1e-10,
               reason="metric computed in Python on each side's predictions; inherits the 1e-12 constant difference")
nb.note("Pipeline fit regressor__sample_weight", "GAP",
        "Moxie Pipeline.fit has no fit-parameter/sample-weight routing (deferred by the pipeline family); "
        "the weighted baseline is fitted on DummyRegressor directly")
nb.note("regression metrics", "ADAPTED", "no moxie.metrics; MSE/MAE/Poisson deviance computed in Python from predictions")
nb.note("OpenML freMTPL2freq", "ADAPTED", "simulated data with the same columns; no download")

Pipeline sample-weight routing is unsupported in Moxie and fails clearly at compile time:

In [6]:
%%mojo pipeline_weighted_fit --expect-fail
from moxie.compose import ColumnTransformer, ColumnEntry, NoTransformer
from moxie.dummy import DummyRegressor
from moxie.pipeline import Pipeline
from numojo import Shape, zeros, ones


def main() raises:
    var columns = ColumnTransformer(
        ColumnEntry[NoTransformer[DType.float64], DType.float64]("numeric", "passthrough", [0]),
        remainder="drop")
    var pipe = Pipeline(columns^, DummyRegressor(strategy="mean"))
    var X = zeros[DType.float64](Shape(3, 1))
    var y = ones[DType.float64](Shape(3))
    _ = pipe.fit(X, y, sample_weight=ones[DType.float64](Shape(3)))

Expected failure (unsupported in Moxie):
compilation failed
/home/strgenix/postdoc/projects/mojods/porting-sklearn/moxie/builds/notebooks/dummy/plot_poisson_regression_non_normal_loss/pipeline_weighted_fit.mojo:14:57: error: invalid call to 'fit': unexpected keyword argument 'sample_weight'
    _ = pipe.fit(X, y, sample_weight=ones[DType.float64](Shape(3)))
        ~~~~~~~~                     ~~~~~~~~~~~~~~~~~~~^~~~~~~~~~
Imported from /home/strgenix/postdoc/projects/mojods/porting-sklearn/moxie/builds/notebooks/dummy/plot_poisson_regression_non_normal_loss/pipeline_weighted_fit.mojo:3:
Imported from /home/strgenix/postdoc/projects/mojods/porting-sklearn/moxie/builds/deps/moxie/pipeline.mojo:10:
/home/strgenix/postdoc/projects/mojods/porting-sklearn/moxie/builds/deps/moxie/_pipeline_sequential.mojo:142:9: note: function declared here
    def fit[
        ^
mojo: error: failed to parse the provided Mojo source module



In [7]:
nb.report()
nb.assert_parity()

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
weighted mean constant_,PASS,"(1, 1)","(1, 1)",2.78e-17,3.46e-16,"rtol=1e-12, atol=0",,"np.average uses pairwise summation, Moxie a sequential weighted sum (20k terms)"
predict (mean_weighted),PASS,"(6600,)","(6600,)",2.78e-17,3.46e-16,"rtol=1e-12, atol=0",,"summation order of the weighted mean, as above"
predict (median_weighted),PASS,"(6600,)","(6600,)",,,exact,0 mismatches,
predict (quantile90_weighted),PASS,"(6600,)","(6600,)",,,exact,0 mismatches,
predict (median),PASS,"(6600,)","(6600,)",,,exact,0 mismatches,
predict (constant),PASS,"(6600,)","(6600,)",,,exact,0 mismatches,
predict(return_std=True) std,PASS,"(6600,)","(6600,)",,,exact,0 mismatches,
"unweighted Pipeline(ColumnTransformer, DummyRegressor) predict",PASS,"(6600,)","(6600,)",0,0,"rtol=1e-12, atol=0",,summation order of the unweighted mean
Pipeline.score with sample_weight (R2),PASS,(),(),2.22e-16,2.16e-12,"rtol=1e-09, atol=0",,R2 ratio of weighted sums; summation order differs
MSE (weighted mean baseline),PASS,(),(),0,0,"rtol=1e-10, atol=0",,metric computed in Python on each side's predictions; inherits the 1e-12 constant difference
